# ============================================
#  Notebook 06 — Metadata & Data Dictionary
#  Memorial Sloan Kettering | Goel Lab
# ============================================

# Notebook 6: Metadata, Data Dictionary & Variable Names

**Purpose:**
1. Generate a **data dictionary** Excel file with standard headers
2. Generate a **variable names** Excel file (concise lookup)
3. Produce a **metadata summary** (dataset-level attributes)

**Data Dictionary Headers:**
| Header | Description |
|--------|-------------|
| Variable Name | Column name as it appears in the dataset |
| Label | Human-readable label |
| Description | Full description of what the variable captures |
| Data Type | Python dtype / conceptual type |
| Valid Values | Permitted coded values or range |
| Source Column | The source/origin column this derives from |
| Domain | Radiology, Pathology, Case-Level, or ID |
| Role | Source, Human Annotator, AI Annotator, Covariate, or ID |
| Missing Code | How missing data is represented |
| Notes | Additional context |

In [ ]:
import os
import re
from pathlib import Path
from datetime import datetime
from collections import OrderedDict

import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/Users/robertjames/Documents/llm_summarization")
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "merged_llm_summary_validation_datasheet_deidentified copy.xlsx"
OUTPUT_DIR = PROJECT_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

data = pd.read_excel(DATA_PATH)
print(f"Dataset: {data.shape[0]} observations x {data.shape[1]} columns")

---
## 1. Element & Domain Definitions

In [ ]:
ELEMENTS = OrderedDict([
    ("Lesion Size", {"source": "lesion_size_status_source", "human": "lesion_size_status_human", "ai": "lesion_size_status_ai"}),
    ("Lesion Laterality", {"source": "laterality_status_source", "human": "laterality_status_human", "ai": "laterality_status_ai"}),
    ("Lesion Location", {"source": "lesion_location_status_source", "human": "lesion_location_status_human", "ai": "lesion_location_status_ai"}),
    ("Calcifications / Asymmetry", {"source": "calcifications_asymmetry_status_source", "human": "calcifications_asymmetry_status_human", "ai": "calcifications_asymmetry_status_ai"}),
    ("Additional Enhancement (MRI)", {"source": "additional_enhancement_mri_status_source", "human": "additional_enhancement_mri_status_human", "ai": "additional_enhancement_mri_status_ai"}),
    ("Extent", {"source": "extent_status_source", "human": "extent_status_human", "ai": "extent_status_ai"}),
    ("Accurate Clip Placement", {"source": "accurate_clip_placement_status_source", "human": "accurate_clip_placement_status_human", "ai": "accurate_clip_placement_status_ai"}),
    ("Workup Recommendation", {"source": "workup_recommendation_status_source", "human": "workup_recommendation_status_human", "ai": "workup_recommendation_status_ai"}),
    ("Lymph Node", {"source": "Lymph node_status_source", "human": "Lymph node_status_human", "ai": "Lymph node_status_ai"}),
    ("Chronology Preserved", {"source": "chronology_preserved_status_source", "human": "chronology_preserved_status_human", "ai": "chronology_preserved_status_ai"}),
    ("Biopsy Method", {"source": "biopsy_method_status_source", "human": "biopsy_method_status_human", "ai": "biopsy_method_status_ai"}),
    ("Invasive Component Size (Pathology)", {"source": "invasive_component_size_pathology_status_source", "human": "invasive_component_size_pathology_status_human", "ai": "invasive_component_size_pathology_status_ai"}),
    ("Histologic Diagnosis", {"source": "histologic_diagnosis_status_source", "human": "histologic_diagnosis_status_human", "ai": "histologic_diagnosis_status_ai"}),
    ("Receptor Status", {"source": "receptor_status_source", "human": "receptor_status_human", "ai": "receptor_status_ai"}),
])

DOMAIN_LOOKUP = {}
for elem in ["Lesion Size", "Lesion Laterality", "Lesion Location",
             "Calcifications / Asymmetry", "Additional Enhancement (MRI)",
             "Extent", "Accurate Clip Placement", "Workup Recommendation",
             "Lymph Node", "Chronology Preserved"]:
    for role_col in ELEMENTS[elem].values():
        DOMAIN_LOOKUP[role_col] = "Radiology"

for elem in ["Biopsy Method", "Invasive Component Size (Pathology)",
             "Histologic Diagnosis", "Receptor Status"]:
    for role_col in ELEMENTS[elem].values():
        DOMAIN_LOOKUP[role_col] = "Pathology"

# Biopsy Method is in both domains
for role_col in ELEMENTS["Biopsy Method"].values():
    DOMAIN_LOOKUP[role_col] = "Radiology / Pathology"

print(f"Domain lookup built: {len(DOMAIN_LOOKUP)} columns mapped")

---
## 2. Build Data Dictionary

In [ ]:
def infer_role(col_name: str) -> str:
    cl = col_name.lower()
    if cl.endswith("_source"):
        return "Source (Ground Truth)"
    elif cl.endswith("_human"):
        return "Human Annotator"
    elif cl.endswith("_ai"):
        return "AI Annotator"
    elif cl in ["surgeon_id"]:
        return "ID"
    elif cl in ["tumor_invasive_dcis", "complex_case_status"]:
        return "Covariate"
    return "Other"

def infer_label(col_name: str) -> str:
    # Find element name
    for elem, cols in ELEMENTS.items():
        for role, cname in cols.items():
            if cname == col_name:
                role_label = {"source": "Source", "human": "Human", "ai": "AI"}[role]
                return f"{elem} ({role_label})"
    # Non-element columns
    labels = {
        "tumor_invasive_dcis": "Tumor Type (Invasive vs DCIS)",
        "complex_case_status": "Complex Case Flag",
        "surgeon_id": "Surgeon Identifier",
    }
    return labels.get(col_name, col_name.replace("_", " ").title())

def infer_description(col_name: str, role: str) -> str:
    for elem, cols in ELEMENTS.items():
        for r, cname in cols.items():
            if cname == col_name:
                if r == "source":
                    return f"Ground truth: whether '{elem}' is present (1) or absent (0) in the source documents."
                elif r == "human":
                    return f"Human annotator extraction status for '{elem}': 1=correctly extracted, 2=omitted, 3=fabricated, N/A=not applicable."
                elif r == "ai":
                    return f"AI (LLM) extraction status for '{elem}': 1=correctly extracted, 2=omitted, 3=fabricated, N/A=not applicable."
    descs = {
        "tumor_invasive_dcis": "Tumor histology classification: 1=Invasive carcinoma, 2=DCIS (ductal carcinoma in situ).",
        "complex_case_status": "Flag indicating whether the case is complex (1) or straightforward (0).",
        "surgeon_id": "De-identified surgeon identifier (e.g., surgeon_001). Links observations to the treating surgeon.",
    }
    return descs.get(col_name, "")

def infer_valid_values(col_name: str, series: pd.Series) -> str:
    for elem, cols in ELEMENTS.items():
        for r, cname in cols.items():
            if cname == col_name:
                if r == "source":
                    return "0 = Absent, 1 = Present"
                else:
                    return "1 = Correct extraction, 2 = Omission, 3 = Fabrication, N/A = Not applicable"
    if col_name == "tumor_invasive_dcis":
        return "1 = Invasive, 2 = DCIS"
    elif col_name == "complex_case_status":
        return "0 = Not complex, 1 = Complex"
    elif col_name == "surgeon_id":
        return f"String ID (e.g., surgeon_001); {series.nunique()} unique values"
    return str(sorted(series.dropna().unique().tolist()))

def infer_missing_code(col_name: str, series: pd.Series) -> str:
    n_miss = int(series.isna().sum())
    has_na_str = False
    if series.dtype == object:
        has_na_str = series.isin(["na", "N/A", "NA"]).any()
    if n_miss == 0 and not has_na_str:
        return "None (no missing)"
    parts = []
    if n_miss > 0:
        parts.append(f"NaN ({n_miss} obs)")
    if has_na_str:
        parts.append("'na'/'N/A' = feature not applicable")
    return "; ".join(parts)

def infer_data_type(col_name: str, series: pd.Series) -> str:
    dtype = str(series.dtype)
    for elem, cols in ELEMENTS.items():
        for r, cname in cols.items():
            if cname == col_name:
                if r == "source":
                    return "Binary integer (0/1)"
                else:
                    return "Categorical (1/2/3/N/A)"
    type_map = {
        "tumor_invasive_dcis": "Binary integer (1/2)",
        "complex_case_status": "Binary integer (0/1)",
        "surgeon_id": "String (categorical)",
    }
    return type_map.get(col_name, dtype)

def infer_source_column(col_name: str) -> str:
    for elem, cols in ELEMENTS.items():
        for r, cname in cols.items():
            if cname == col_name and r in ["human", "ai"]:
                return cols["source"]
    return ""

def infer_notes(col_name: str) -> str:
    for elem, cols in ELEMENTS.items():
        for r, cname in cols.items():
            if cname == col_name:
                if r == "source":
                    return "Ground truth derived from source documents"
                elif r == "human":
                    return "Extracted by human annotator; compared against source"
                elif r == "ai":
                    return "Extracted by LLM (base model); compared against source"
    notes = {
        "tumor_invasive_dcis": "Case-level covariate; used for stratification",
        "complex_case_status": "Case-level covariate; may affect extraction difficulty",
        "surgeon_id": "De-identified; used for clustering/random effects",
    }
    return notes.get(col_name, "")

print("Data dictionary builder functions defined.")

In [ ]:
# Build data dictionary
dict_rows = []
for col in data.columns:
    series = data[col]
    dict_rows.append({
        "Variable Name": col,
        "Label": infer_label(col),
        "Description": infer_description(col, infer_role(col)),
        "Data Type": infer_data_type(col, series),
        "Valid Values": infer_valid_values(col, series),
        "Source Column": infer_source_column(col),
        "Domain": DOMAIN_LOOKUP.get(col, "Case-Level" if col in ["tumor_invasive_dcis", "complex_case_status"] else ("ID" if col == "surgeon_id" else "")),
        "Role": infer_role(col),
        "Missing Code": infer_missing_code(col, series),
        "Notes": infer_notes(col),
    })

df_dict = pd.DataFrame(dict_rows)
print(f"Data dictionary: {df_dict.shape[0]} variables")
df_dict

---
## 3. Build Variable Names File

In [ ]:
# Concise variable lookup
var_rows = []
for i, col in enumerate(data.columns):
    series = data[col]
    var_rows.append({
        "Position": i + 1,
        "Variable Name": col,
        "Label": infer_label(col),
        "Data Type": infer_data_type(col, series),
        "Domain": DOMAIN_LOOKUP.get(col, "Case-Level" if col in ["tumor_invasive_dcis", "complex_case_status"] else ("ID" if col == "surgeon_id" else "")),
        "Role": infer_role(col),
        "N Unique": int(series.nunique()),
        "N Missing": int(series.isna().sum()),
        "Pct Missing": round(series.isna().mean() * 100, 2),
    })

df_vars = pd.DataFrame(var_rows)
print(f"Variable names file: {df_vars.shape[0]} variables")
df_vars

---
## 4. Build Metadata Summary

In [ ]:
# Dataset-level metadata
metadata = OrderedDict([
    ("Dataset Name", "Merged LLM Summary Validation Datasheet (Deidentified)"),
    ("Source File", str(DATA_PATH.name)),
    ("Date Generated", datetime.now().strftime("%Y-%m-%d %H:%M:%S")),
    ("N Observations", data.shape[0]),
    ("N Variables", data.shape[1]),
    ("N Elements (Features)", len(ELEMENTS)),
    ("N Radiology Elements", sum(1 for v in DOMAIN_LOOKUP.values() if "Radiology" in v)),
    ("N Pathology Elements", sum(1 for v in DOMAIN_LOOKUP.values() if "Pathology" in v)),
    ("Annotator Roles", "Source (Ground Truth), Human Annotator, AI (LLM) Annotator"),
    ("Source Coding", "0 = Feature absent in source document, 1 = Feature present"),
    ("Annotator Coding", "1 = Correct extraction, 2 = Omission, 3 = Fabrication, N/A = Not applicable"),
    ("Covariates", "tumor_invasive_dcis, complex_case_status"),
    ("ID Variables", "surgeon_id"),
    ("Total Missing Cells", int(data.isna().sum().sum())),
    ("Overall Missingness (%)", round(data.isna().sum().sum() / (data.shape[0] * data.shape[1]) * 100, 2)),
    ("Tumor Invasive Count", int((data["tumor_invasive_dcis"] == 1).sum())),
    ("Tumor DCIS Count", int((data["tumor_invasive_dcis"] == 2).sum())),
    ("Complex Case Count", int((data["complex_case_status"] == 1).sum())),
    ("Non-Complex Case Count", int((data["complex_case_status"] == 0).sum())),
    ("N Unique Surgeons", int(data["surgeon_id"].nunique())),
    ("Project", "Prompt-Technique Evaluation for Feature-Level Human vs LLM Validation (Base Model)"),
    ("Repository", "https://github.com/Robertjam954/AI-Jupyter-Notebook.git"),
])

df_metadata = pd.DataFrame(list(metadata.items()), columns=["Attribute", "Value"])
print("Dataset Metadata:")
df_metadata

In [ ]:
# Per-element metadata summary
element_meta_rows = []
for elem, cols in ELEMENTS.items():
    s_col = cols["source"]
    h_col = cols["human"]
    a_col = cols["ai"]
    domain = DOMAIN_LOOKUP.get(s_col, "")
    n_present = int((data[s_col] == 1).sum()) if s_col in data.columns else 0
    n_absent = int((data[s_col] == 0).sum()) if s_col in data.columns else 0
    h_miss = int(data[h_col].isna().sum()) if h_col in data.columns else data.shape[0]
    a_miss = int(data[a_col].isna().sum()) if a_col in data.columns else data.shape[0]
    element_meta_rows.append({
        "Element": elem,
        "Domain": domain,
        "Source Column": s_col,
        "Human Column": h_col,
        "AI Column": a_col,
        "N Source Present (1)": n_present,
        "N Source Absent (0)": n_absent,
        "Prevalence (%)": round(n_present / data.shape[0] * 100, 1),
        "Human Missing (N)": h_miss,
        "AI Missing (N)": a_miss,
    })

df_element_meta = pd.DataFrame(element_meta_rows)
print("\nPer-Element Metadata:")
df_element_meta

---
## 5. Export to Excel

In [ ]:
# --- Data Dictionary Excel ---
dict_path = OUTPUT_DIR / "data_dictionary.xlsx"
with pd.ExcelWriter(dict_path, engine="openpyxl") as writer:
    df_dict.to_excel(writer, sheet_name="Data Dictionary", index=False)
    df_element_meta.to_excel(writer, sheet_name="Element Summary", index=False)
    df_metadata.to_excel(writer, sheet_name="Dataset Metadata", index=False)
print(f"Data Dictionary saved: {dict_path}")

# --- Variable Names Excel ---
var_path = OUTPUT_DIR / "variable_names.xlsx"
with pd.ExcelWriter(var_path, engine="openpyxl") as writer:
    df_vars.to_excel(writer, sheet_name="Variable Names", index=False)
print(f"Variable Names saved: {var_path}")

In [ ]:
# Format the Excel files with column widths and header styling
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

def style_excel(filepath, sheet_configs=None):
    """Apply consistent formatting to an Excel workbook."""
    wb = load_workbook(filepath)
    header_font = Font(bold=True, color="FFFFFF", size=11)
    header_fill = PatternFill(start_color="2C3E50", end_color="2C3E50", fill_type="solid")
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    cell_align = Alignment(vertical="top", wrap_text=True)
    thin_border = Border(
        left=Side(style="thin"),
        right=Side(style="thin"),
        top=Side(style="thin"),
        bottom=Side(style="thin"),
    )

    for ws in wb.worksheets:
        # Auto-size columns (approximate)
        for col in ws.columns:
            max_length = 0
            col_letter = col[0].column_letter
            for cell in col:
                try:
                    if cell.value:
                        max_length = max(max_length, len(str(cell.value)))
                except:
                    pass
            adjusted_width = min(max_length + 4, 50)
            ws.column_dimensions[col_letter].width = adjusted_width

        # Style header row
        for cell in ws[1]:
            cell.font = header_font
            cell.fill = header_fill
            cell.alignment = header_align
            cell.border = thin_border

        # Style data rows
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = cell_align
                cell.border = thin_border

        # Freeze header row
        ws.freeze_panes = "A2"

    wb.save(filepath)
    print(f"  Styled: {filepath}")

style_excel(dict_path)
style_excel(var_path)
print("\nDone. All Excel files styled and saved.")

---
## 6. Quick Validation

In [ ]:
# Verify outputs
print("=" * 60)
print("OUTPUT FILES")
print("=" * 60)
for f in [dict_path, var_path]:
    if f.exists():
        size_kb = f.stat().st_size / 1024
        print(f"  {f.name:40s} {size_kb:7.1f} KB")
    else:
        print(f"  {f.name:40s} MISSING")

print(f"\nData Dictionary:")
print(f"  Sheets: Data Dictionary ({len(df_dict)} rows), Element Summary ({len(df_element_meta)} rows), Dataset Metadata ({len(df_metadata)} rows)")
print(f"  Headers: {list(df_dict.columns)}")
print(f"\nVariable Names:")
print(f"  Sheets: Variable Names ({len(df_vars)} rows)")
print(f"  Headers: {list(df_vars.columns)}")